# RUNG 12 (structure) — ColabFold pMHC poses → MEASURED exposure → certified relay targets (T4 GPU)

The first RUNG-12 run's ESMFold failed (openfold build + single-seq can't dock a 9-mer). This redoes the **structural** arm with **ColabFold / AlphaFold2-multimer** (MSA-guided → actually docks the peptide; license-free; GPU). It folds the **HLA α1α2 groove + peptide** for the **non-`clean` handles** (clean handles have β=0 regardless of structure — no need to fold them), measures the **TCR-facing exposure (RSA)** of the mutated residue, and feeds the measured `E` into `scripts/37`'s β scoring → re-runs the bridge with real structural exposure.

**Validated:** the RSA analysis was checked against a real HLA-A*02:01 crystal (1HHK) — it correctly flags buried anchors (P2/P9 ≈ 0.02) vs TCR-facing residues (P4–6 ≈ 0.5). So *given a reliable pose*, exposure is read correctly.

**Honest:** AF2 docks pMHC-I reasonably but a 1-residue mut/WT change barely moves the backbone → the signal is the mutated residue's **exposure**, not RMSD. β stays a proxy → top handles = prioritised wet-lab hypotheses. **If the ColabFold install drifts (it does), report the error — the analysis + fallbacks are solid, so a partial fold still yields a valid result.** Use **T4 GPU**, resumable Drive PDBs.

In [ ]:
#@title Cell 1 — clone / pull repo
import os
from pathlib import Path
REPO = Path('/content/cancer-recon-apoptosis')
if REPO.exists():
    !cd {REPO} && rm -f runs/logs/*.log && git fetch origin && git reset --hard origin/main
else:
    !git clone https://github.com/AnshumanAtrey/cancer-recon-apoptosis.git {REPO}
os.chdir(REPO)
!git log --oneline -1
print('[CELL 1] ✓')

In [ ]:
#@title Cell 2 — mount Drive (resumable PDBs) + PREP (handles, grooves, multimer FASTAs)
import os, sys
try:
    from google.colab import drive; drive.mount('/content/drive')
    cd = '/content/drive/MyDrive/cancer-recon'; os.makedirs(cd, exist_ok=True)
    os.environ['RUNG12_WORK'] = f'{cd}/rung12_work'; os.makedirs(os.environ['RUNG12_WORK'], exist_ok=True)
    print('[CELL 2] Drive mounted; work dir', os.environ['RUNG12_WORK'])
except Exception as e:
    os.environ['RUNG12_WORK'] = '/content/rung12_work'; os.makedirs('/content/rung12_work', exist_ok=True)
    print('[CELL 2] Drive NOT mounted (', type(e).__name__, ') — PDBs in /content (lost on disconnect)')
os.environ['RUNG12_N'] = '32'
from scripts.runlog import new_log, run_logged
RUNLOG = new_log('rung12_structure', repo_dir='.')
run_logged([sys.executable,'-u','scripts/37_pmhc_discriminability.py','prep'], RUNLOG)

In [ ]:
#@title Cell 3 — VALIDATE scoring logic (synthetic, instant)
import sys
from scripts.runlog import run_logged
rc = run_logged([sys.executable,'-u','scripts/37_pmhc_discriminability.py','selftest'], RUNLOG)
print('[CELL 3]', '✓ validated' if rc==0 else '✗ FAILED — stop here')

In [ ]:
#@title Cell 4 — install ColabFold (AlphaFold2-multimer) + Biopython
#@markdown Install can take ~5 min and occasionally drifts with CUDA — if `colabfold_batch` isn't found,
#@markdown tell me the error and I'll patch the pin. Biopython is for the RSA analysis.
!pip install -q biopython
!pip install -q "colabfold[alphafold] @ git+https://github.com/sokrypton/ColabFold@v1.5.5" 2>&1 | tail -3
# Colab GPU needs a CUDA-matched jax (ColabFold pulls CPU jax by default):
!pip install -q --upgrade "jax[cuda12]" 2>&1 | tail -2 || echo '(jax cuda note — may already be present)'
import shutil
print('[CELL 4]', 'colabfold_batch:', shutil.which('colabfold_batch') or 'NOT FOUND — report this')

In [ ]:
#@title Cell 5 — FOLD non-clean handles (ColabFold, MSA via MMseqs2, resumable)
#@markdown Only `tcr_dependent`/`anchor` handles are folded (clean → β=0 regardless). Each FASTA is
#@markdown `groove:peptide` (multimer). Rank-1 model is copied to `<id>_{mut,wt}.pdb` for analysis.
import os, json, glob, shutil, subprocess
WORK = os.environ['RUNG12_WORK']
man = json.load(open('runs/rung12_pmhc/rung12_manifest.json'))
need = [h for h in man['handles'] if h.get('tier') and h['tier'] != 'clean' and 'groove' in h]
print(f'[CELL 5] folding {len(need)} non-clean handles x2 (mut+WT); clean handles skipped (beta=0)')
indir = f'{WORK}/cf_in'; outdir = f'{WORK}/cf_out'; os.makedirs(indir, exist_ok=True); os.makedirs(outdir, exist_ok=True)
todo = []
for h in need:
    for tag in ('mut','wt'):
        dst = f'{WORK}/{h["id"]}_{tag}.pdb'
        if os.path.exists(dst):
            continue
        src = f'{WORK}/{h["id"]}_{tag}.fasta'
        if os.path.exists(src): shutil.copy(src, f'{indir}/{h["id"]}_{tag}.fasta'); todo.append((h['id'],tag))
if todo:
    cmd = ['colabfold_batch', indir, outdir, '--num-models','1','--num-recycle','3',
           '--msa-mode','mmseqs2_uniref_env','--model-type','alphafold2_multimer_v3']
    print('  running:', ' '.join(cmd)); subprocess.run(cmd)
# map ColabFold rank-1 outputs -> <id>_<tag>.pdb (what scripts/37 analyze reads)
mapped = 0
for h in need:
    for tag in ('mut','wt'):
        dst = f'{WORK}/{h["id"]}_{tag}.pdb'
        if os.path.exists(dst): mapped += 1; continue
        hits = sorted(glob.glob(f'{outdir}/{h["id"]}_{tag}_*rank_001*.pdb'))
        if hits: shutil.copy(hits[0], dst); mapped += 1
print(f'[CELL 5] ✓ {mapped} PDBs ready in {WORK} (resumable; re-run to continue if disconnected)')

In [ ]:
#@title Cell 6 — ANALYZE: measured exposure E -> beta -> ranked targets + bridge
import sys, os, json
from scripts.runlog import run_logged
rc = run_logged([sys.executable,'-u','scripts/37_pmhc_discriminability.py','analyze'], RUNLOG)
print('[CELL 6]', '✓ done' if rc==0 else f'✗ exit {rc}')
from IPython.display import Image, display
if os.path.exists('runs/rung12_pmhc/rung12_pmhc.png'): display(Image('runs/rung12_pmhc/rung12_pmhc.png'))
if os.path.exists('runs/rung12_pmhc/rung12_pmhc.json'):
    d = json.load(open('runs/rung12_pmhc/rung12_pmhc.json'))
    print('with-structure:', d['n_with_structure'], '/', d['n_handles'],
          '| per-cell-safe', d['n_per_cell_safe'], '| relay-safe', d['n_relay_safe'],
          '| unlocked', d['n_unlocked_by_relay'])

In [ ]:
#@title Cell 7 — bundle ONE zip + download
import os, glob, zipfile
from scripts.runlog import finalize
finalize(RUNLOG, download=False)
stem = os.path.basename(str(RUNLOG)).replace('rung12_structure_','').replace('.log','')
b = f'/content/rung12_run_{stem}.zip'
ps = sorted(set(glob.glob('runs/rung12_pmhc/*.png')+glob.glob('runs/rung12_pmhc/*.json')+[str(RUNLOG)]))
with zipfile.ZipFile(b, 'w', zipfile.ZIP_DEFLATED) as z:
    for x in ps:
        if os.path.exists(x): z.write(x, arcname=os.path.basename(x)); print(' bundled', x)
print('[CELL 7] ->', b)
try:
    from google.colab import files; files.download(b)
except Exception as e: print('(download skipped:', e, ')')